# NB-05: Estratégia Final Otimizada

**Objetivo**: Combinar os melhores achados dos NB-01 a NB-04 numa estratégia final validada.

## Pipeline:
1. **Walk-Forward Validation** - Treino 2022-2024, teste 2025-2026
2. **Grid Search** - Otimizar trigger, pattern, target, stops simultaneamente
3. **Monte Carlo Robusto** - 10.000 simulações com bootstrap
4. **Análise de Regimes** - Performance por período/volatilidade
5. **Configuração Final** - Parâmetros prontos para o bot
6. **Exportar JSON** - Config machine-readable para `config/`

In [ ]:
import sys
sys.path.insert(0, '..')

# Carregar framework do backtester (NB-02)
%run 12_backtester.ipynb

In [ ]:
import json
from itertools import product
from pathlib import Path

df = load_raw_data()
multipliers = df['multiplicador'].values

# Separar treino (2022-2024) e teste (2025-2026)
df['date'] = pd.to_datetime(df['date'])
mask_train = df['date'] < '2025-01-01'
mask_test = df['date'] >= '2025-01-01'

mult_train = multipliers[mask_train.values]
mult_test = multipliers[mask_test.values]

print(f"Total:  {len(multipliers):>10,} rounds")
print(f"Treino: {len(mult_train):>10,} rounds (Dez/2022 - Dez/2024)")
print(f"Teste:  {len(mult_test):>10,} rounds (Jan/2025 - Fev/2026)")
print(f"")
print(f"% LOW treino: {(mult_train < LOW_THRESHOLD).mean()*100:.1f}%")
print(f"% LOW teste:  {(mult_test < LOW_THRESHOLD).mean()*100:.1f}%")

## 1. Estratégias Candidatas (Top do NB-03)

Reimplementamos as melhores estratégias do torneio para o grid search.

In [ ]:
# Reimportar estratégias do NB-03
# (executadas via %run do backtester; reimplementar aqui para ficar standalone)

class AdaptiveTriggerStrategy(BettingStrategy):
    """Trigger adaptativo baseado na volatilidade recente."""

    def __init__(self, base_trigger=6, target=2.0, window=200,
                 min_trigger=4, max_trigger=10, pattern=None):
        self.base_trigger = base_trigger
        self.target = target
        self.window = window
        self.min_trigger = min_trigger
        self.max_trigger = max_trigger
        self.pattern = pattern or [1, 2, 4]
        self.reset()

    @property
    def name(self):
        p = '/'.join(str(x) for x in self.pattern)
        return f"Adaptativo {p} (T{self.base_trigger}, w{self.window})"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.current_dobra = 0
        self.low_history = []
        self.current_trigger = self.base_trigger

    def _update_trigger(self):
        if len(self.low_history) < self.window:
            return
        recent = self.low_history[-self.window:]
        pct_low = np.mean(recent)
        if pct_low > 0.58:
            self.current_trigger = min(self.base_trigger + 2, self.max_trigger)
        elif pct_low > 0.55:
            self.current_trigger = self.base_trigger + 1
        elif pct_low < 0.50:
            self.current_trigger = max(self.base_trigger - 1, self.min_trigger)
        else:
            self.current_trigger = self.base_trigger

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD
        self.low_history.append(1 if is_low else 0)
        self._update_trigger()

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.current_trigger:
                self.in_sequence = True
                self.current_dobra = 0
                mult = self.pattern[0]
                return BetDecision(True, base_bet * mult, self.target)
            return BetDecision(False)

        if multiplier >= self.target:
            self.in_sequence = False
            self.consecutive_lows = 0
            return BetDecision(False)
        else:
            self.current_dobra += 1
            if self.current_dobra >= len(self.pattern):
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            mult = self.pattern[self.current_dobra]
            return BetDecision(True, base_bet * mult, self.target)


class DAlembertStrategy(BettingStrategy):
    """D'Alembert: +1 unidade após perda, -1 após ganho."""

    def __init__(self, trigger=6, target=2.0, max_units=10):
        self.trigger = trigger
        self.target = target
        self.max_units = max_units
        self.reset()

    @property
    def name(self):
        return f"D'Alembert (T{self.trigger}, max{self.max_units}u)"

    def reset(self):
        self.consecutive_lows = 0
        self.in_sequence = False
        self.units = 1

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD

        if not self.in_sequence:
            if is_low:
                self.consecutive_lows += 1
            else:
                self.consecutive_lows = 0
            if self.consecutive_lows >= self.trigger:
                self.in_sequence = True
                self.units = 1
                return BetDecision(True, base_bet, self.target)
            return BetDecision(False)

        if multiplier >= self.target:
            self.units = max(1, self.units - 1)
            if self.units == 1:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.units, self.target)
        else:
            self.units += 1
            if self.units > self.max_units:
                self.in_sequence = False
                self.consecutive_lows = 0
                return BetDecision(False)
            return BetDecision(True, base_bet * self.units, self.target)


class KellyCriterionStrategy(BettingStrategy):
    """Kelly Criterion: aposta ótima baseada na prob. recente."""

    def __init__(self, trigger=6, target=2.0, window=500, fraction=0.5):
        self.trigger = trigger
        self.target = target
        self.window = window
        self.fraction = fraction
        self.reset()

    @property
    def name(self):
        return f"Kelly {self.fraction:.0%} (T{self.trigger}, w{self.window})"

    def reset(self):
        self.consecutive_lows = 0
        self.history = []

    def on_round(self, multiplier, bankroll, base_bet, round_idx):
        is_low = multiplier < LOW_THRESHOLD
        self.history.append(multiplier)

        if is_low:
            self.consecutive_lows += 1
        else:
            self.consecutive_lows = 0

        if self.consecutive_lows >= self.trigger:
            self.consecutive_lows = 0
            recent = self.history[-self.window:] if len(self.history) >= self.window else self.history
            p_win = np.mean(np.array(recent) >= self.target)
            b = self.target - 1
            q = 1 - p_win
            kelly_pct = (p_win * b - q) / b if b > 0 else 0
            kelly_pct = max(0, kelly_pct) * self.fraction
            if kelly_pct > 0:
                bet = bankroll * kelly_pct
                return BetDecision(True, bet, self.target)

        return BetDecision(False)


print("Estratégias carregadas: Martingale, Adaptativo, D'Alembert, Kelly")

## 2. Grid Search sobre Dados de Treino (2022-2024)

Testar combinações de parâmetros e ranquear por Sharpe ratio.

In [ ]:
# Parâmetros do grid
triggers = [5, 6, 7, 8]
patterns = [
    ([1, 2], '1/2'),
    ([1, 2, 4], '1/2/4'),
    ([1, 2, 1, 2], '1/2+1/2'),
    ([1, 2, 4, 1, 2, 4], '1/2/4+1/2/4'),
    ([1, 2, 4, 8], '1/2/4/8'),
    ([1, 1, 2, 2, 4], '1/1/2/2/4'),
]
targets = [1.8, 2.0, 2.2, 2.5]
stop_losses = [0.30, 0.50, 0.80]
stop_gains = [0.10, 0.20, 0.30]

grid_results = []
total = len(triggers) * len(patterns) * len(targets) * len(stop_losses) * len(stop_gains)
print(f"Grid total: {total} combinações")
print(f"Rodando sobre {len(mult_train):,} rounds de treino...")
print()

count = 0
for trigger in triggers:
    for (pattern, pattern_name) in patterns:
        for target in targets:
            for sl in stop_losses:
                for sg in stop_gains:
                    cfg = BankrollConfig(
                        initial_bankroll=1000,
                        base_bet_pct=0.0167,
                        compound=False,
                        stop_loss_pct=sl,
                        stop_gain_pct=sg,
                    )
                    strat = MartingaleStrategy(
                        trigger=trigger, target=target, pattern=pattern
                    )
                    r = backtest(strat, mult_train, cfg)

                    grid_results.append({
                        'trigger': trigger,
                        'pattern': pattern_name,
                        'target': target,
                        'stop_loss': sl,
                        'stop_gain': sg,
                        'profit': r.total_profit,
                        'profit_pct': r.total_profit_pct,
                        'win_rate': r.win_rate,
                        'max_dd': r.max_drawdown_pct,
                        'sharpe': r.sharpe_ratio,
                        'profit_factor': r.profit_factor,
                        'total_bets': r.total_bets,
                        'max_consec_loss': r.max_consecutive_losses,
                    })

                    count += 1
                    if count % 100 == 0:
                        print(f"  {count}/{total} ({count/total*100:.0f}%)")

grid_df = pd.DataFrame(grid_results)
print(f"\nGrid completo: {len(grid_df)} resultados")

In [ ]:
# Top 20 por Sharpe Ratio
top20 = grid_df.nlargest(20, 'sharpe')

print("TOP 20 CONFIGURAÇÕES POR SHARPE RATIO (dados de treino)")
print("=" * 110)
print(f"{'#':>3} {'Trigger':>7} {'Pattern':>12} {'Target':>7} {'SL%':>5} {'SG%':>5} "
      f"{'Sharpe':>8} {'Lucro':>10} {'WR%':>6} {'DD%':>7} {'PF':>6} {'Apostas':>8}")
print("-" * 110)

for i, (_, row) in enumerate(top20.iterrows(), 1):
    print(f"{i:>3} T{row['trigger']:>5} {row['pattern']:>12} {row['target']:>6.1f}x "
          f"{row['stop_loss']*100:>4.0f}% {row['stop_gain']*100:>4.0f}% "
          f"{row['sharpe']:>+8.4f} R${row['profit']:>+8.0f} {row['win_rate']:>5.0f}% "
          f"{row['max_dd']:>6.1f}% {row['profit_factor']:>5.2f} {row['total_bets']:>8}")

In [ ]:
# Heatmaps: Sharpe por Trigger x Pattern (fixando melhor target/SL/SG)
best = top20.iloc[0]
best_target = best['target']
best_sl = best['stop_loss']
best_sg = best['stop_gain']

print(f"Melhor config do treino: T{best['trigger']} | {best['pattern']} | "
      f"{best_target}x | SL {best_sl*100:.0f}% | SG {best_sg*100:.0f}%")
print(f"Sharpe: {best['sharpe']:.4f} | Lucro: R${best['profit']:+.0f} | "
      f"WR: {best['win_rate']:.0f}% | DD: {best['max_dd']:.1f}%")
print()

# Heatmap Trigger x Pattern
fig, axes = plt.subplots(1, 3, figsize=(20, 5))

# Filter para melhor target
sub = grid_df[(grid_df['target'] == best_target) & 
              (grid_df['stop_loss'] == best_sl) &
              (grid_df['stop_gain'] == best_sg)]

pivot = sub.pivot_table(values='sharpe', index='trigger', columns='pattern', aggfunc='mean')
im1 = axes[0].imshow(pivot.values, cmap='RdYlGn', aspect='auto')
axes[0].set_xticks(range(len(pivot.columns)))
axes[0].set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels([f'T{t}' for t in pivot.index])
axes[0].set_title(f'Sharpe: Trigger x Pattern\n(target={best_target}x)')
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        if not np.isnan(val):
            axes[0].text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im1, ax=axes[0])

# Heatmap Target x SL
sub2 = grid_df[(grid_df['trigger'] == best['trigger']) &
               (grid_df['pattern'] == best['pattern']) &
               (grid_df['stop_gain'] == best_sg)]
pivot2 = sub2.pivot_table(values='sharpe', index='stop_loss', columns='target', aggfunc='mean')
im2 = axes[1].imshow(pivot2.values, cmap='RdYlGn', aspect='auto')
axes[1].set_xticks(range(len(pivot2.columns)))
axes[1].set_xticklabels([f'{t}x' for t in pivot2.columns])
axes[1].set_yticks(range(len(pivot2.index)))
axes[1].set_yticklabels([f'{sl*100:.0f}%' for sl in pivot2.index])
axes[1].set_title(f'Sharpe: Target x Stop Loss\n(T{int(best["trigger"])}, {best["pattern"]})')
for i in range(len(pivot2.index)):
    for j in range(len(pivot2.columns)):
        val = pivot2.values[i, j]
        if not np.isnan(val):
            axes[1].text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im2, ax=axes[1])

# Heatmap SG x SL
sub3 = grid_df[(grid_df['trigger'] == best['trigger']) &
               (grid_df['pattern'] == best['pattern']) &
               (grid_df['target'] == best_target)]
pivot3 = sub3.pivot_table(values='sharpe', index='stop_loss', columns='stop_gain', aggfunc='mean')
im3 = axes[2].imshow(pivot3.values, cmap='RdYlGn', aspect='auto')
axes[2].set_xticks(range(len(pivot3.columns)))
axes[2].set_xticklabels([f'{sg*100:.0f}%' for sg in pivot3.columns])
axes[2].set_yticks(range(len(pivot3.index)))
axes[2].set_yticklabels([f'{sl*100:.0f}%' for sl in pivot3.index])
axes[2].set_title(f'Sharpe: Stop Gain x Stop Loss\n(T{int(best["trigger"])}, {best["pattern"]}, {best_target}x)')
for i in range(len(pivot3.index)):
    for j in range(len(pivot3.columns)):
        val = pivot3.values[i, j]
        if not np.isnan(val):
            axes[2].text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im3, ax=axes[2])

plt.tight_layout()
plt.show()

## 3. Walk-Forward Validation

Testar as Top 5 configurações do treino sobre os dados **nunca vistos** (2025-2026).
Se a performance degrada muito, há overfitting.

In [ ]:
# Top 5 do treino
top5_configs = top20.head(5).to_dict('records')

print("WALK-FORWARD: Treino (2022-2024) → Teste (2025-2026)")
print("=" * 100)
print(f"{'Config':>50} {'Sharpe TREINO':>14} {'Sharpe TESTE':>14} {'Degradação':>12}")
print("-" * 100)

wf_results = []

for cfg_dict in top5_configs:
    # Converter pattern string de volta para lista
    pattern_map = {
        '1/2': [1, 2],
        '1/2/4': [1, 2, 4],
        '1/2+1/2': [1, 2, 1, 2],
        '1/2/4+1/2/4': [1, 2, 4, 1, 2, 4],
        '1/2/4/8': [1, 2, 4, 8],
        '1/1/2/2/4': [1, 1, 2, 2, 4],
    }
    pattern = pattern_map[cfg_dict['pattern']]
    
    bankroll_cfg = BankrollConfig(
        initial_bankroll=1000,
        base_bet_pct=0.0167,
        compound=False,
        stop_loss_pct=cfg_dict['stop_loss'],
        stop_gain_pct=cfg_dict['stop_gain'],
    )

    # Treino
    strat_train = MartingaleStrategy(
        trigger=int(cfg_dict['trigger']),
        target=cfg_dict['target'],
        pattern=pattern,
    )
    r_train = backtest(strat_train, mult_train, bankroll_cfg)

    # Teste
    strat_test = MartingaleStrategy(
        trigger=int(cfg_dict['trigger']),
        target=cfg_dict['target'],
        pattern=pattern,
    )
    r_test = backtest(strat_test, mult_test, bankroll_cfg)

    degradation = 0
    if r_train.sharpe_ratio != 0:
        degradation = (r_test.sharpe_ratio - r_train.sharpe_ratio) / abs(r_train.sharpe_ratio) * 100

    config_label = (f"T{int(cfg_dict['trigger'])} {cfg_dict['pattern']} "
                    f"{cfg_dict['target']}x SL{cfg_dict['stop_loss']*100:.0f} "
                    f"SG{cfg_dict['stop_gain']*100:.0f}")

    print(f"{config_label:>50} {r_train.sharpe_ratio:>+14.4f} "
          f"{r_test.sharpe_ratio:>+14.4f} {degradation:>+11.1f}%")

    wf_results.append({
        'config': cfg_dict,
        'config_label': config_label,
        'train_sharpe': r_train.sharpe_ratio,
        'test_sharpe': r_test.sharpe_ratio,
        'train_profit': r_train.total_profit,
        'test_profit': r_test.total_profit,
        'train_dd': r_train.max_drawdown_pct,
        'test_dd': r_test.max_drawdown_pct,
        'train_wr': r_train.win_rate,
        'test_wr': r_test.win_rate,
        'degradation': degradation,
        'r_train': r_train,
        'r_test': r_test,
    })

In [ ]:
# Visualizar walk-forward
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

colors_list = [COLORS['blue'], COLORS['green'], COLORS['red'],
               COLORS['yellow'], COLORS['purple']]

# Equity curves treino
for i, wf in enumerate(wf_results):
    r = wf['r_train']
    label = f"{wf['config_label']} (S={wf['train_sharpe']:+.3f})"
    axes[0].plot(r.equity_curve, color=colors_list[i], linewidth=1, alpha=0.8, label=label)
axes[0].axhline(y=1000, color=COLORS['dim'], linestyle='--', alpha=0.3)
axes[0].set_title('TREINO (2022-2024)')
axes[0].set_ylabel('Banca (R$)')
axes[0].legend(fontsize=8)

# Equity curves teste
for i, wf in enumerate(wf_results):
    r = wf['r_test']
    label = f"{wf['config_label']} (S={wf['test_sharpe']:+.3f})"
    axes[1].plot(r.equity_curve, color=colors_list[i], linewidth=1, alpha=0.8, label=label)
axes[1].axhline(y=1000, color=COLORS['dim'], linestyle='--', alpha=0.3)
axes[1].set_title('TESTE (2025-2026) - Dados Nunca Vistos')
axes[1].set_ylabel('Banca (R$)')
axes[1].set_xlabel('Round')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Selecionar melhor config out-of-sample
best_wf = max(wf_results, key=lambda x: x['test_sharpe'])
print(f"\nMELHOR CONFIG OUT-OF-SAMPLE: {best_wf['config_label']}")
print(f"  Sharpe treino: {best_wf['train_sharpe']:+.4f}")
print(f"  Sharpe teste:  {best_wf['test_sharpe']:+.4f}")
print(f"  Lucro treino:  R${best_wf['train_profit']:+.0f}")
print(f"  Lucro teste:   R${best_wf['test_profit']:+.0f}")
print(f"  DD treino:     {best_wf['train_dd']:.1f}%")
print(f"  DD teste:      {best_wf['test_dd']:.1f}%")

## 4. Comparação com Estratégias Alternativas (Walk-Forward)

Testar Adaptativo, D'Alembert e Kelly no mesmo split para comparar.

In [ ]:
# Usar os parâmetros do melhor walk-forward
bwf = best_wf['config']
best_trigger = int(bwf['trigger'])
best_target_val = bwf['target']
best_pattern = pattern_map[bwf['pattern']]
best_sl_val = bwf['stop_loss']
best_sg_val = bwf['stop_gain']

bankroll_cfg = BankrollConfig(
    initial_bankroll=1000,
    base_bet_pct=0.0167,
    compound=False,
    stop_loss_pct=best_sl_val,
    stop_gain_pct=best_sg_val,
)

# Estratégias alternativas nos dados de TESTE
alt_strategies = [
    MartingaleStrategy(trigger=best_trigger, target=best_target_val, pattern=best_pattern),
    AdaptiveTriggerStrategy(base_trigger=best_trigger, target=best_target_val, pattern=best_pattern),
    DAlembertStrategy(trigger=best_trigger, target=best_target_val, max_units=8),
    KellyCriterionStrategy(trigger=best_trigger, target=best_target_val, fraction=0.5),
]

alt_results_test = []
print("COMPARAÇÃO DE ESTRATÉGIAS NO PERÍODO DE TESTE (2025-2026)")
print("=" * 90)

for strat in alt_strategies:
    r = backtest(strat, mult_test, bankroll_cfg)
    alt_results_test.append(r)
    print(f"{r.strategy_name:45s} | Sharpe: {r.sharpe_ratio:+.4f} | "
          f"Lucro: R${r.total_profit:+.0f} | WR: {r.win_rate:.0f}% | DD: {r.max_drawdown_pct:.1f}%")

plot_comparison(alt_results_test, 'Estratégias no Período de Teste (2025-2026)')

## 5. Monte Carlo Robusto (10.000 Simulações)

Bootstrap com resampling dos rounds reais para estimar:
- Distribuição do lucro esperado
- Intervalo de confiança (95%)
- Probabilidade de ruin
- Percentis de drawdown

In [ ]:
n_simulations = 10_000
sim_session_size = 3000  # ~equivale a 1 dia intenso de jogo

print(f"Monte Carlo: {n_simulations:,} simulações de {sim_session_size} rounds")
print(f"Config: T{best_trigger} {bwf['pattern']} {best_target_val}x SL{best_sl_val*100:.0f}% SG{best_sg_val*100:.0f}%")
print("Rodando...")

mc_profits = []
mc_max_dds = []
mc_final_bankrolls = []
mc_win_rates = []
mc_bets_count = []
ruin_count = 0

for sim in range(n_simulations):
    # Bootstrap: amostrar bloco aleatório dos dados completos
    start = np.random.randint(0, len(multipliers) - sim_session_size)
    chunk = multipliers[start:start + sim_session_size]

    strat = MartingaleStrategy(
        trigger=best_trigger, target=best_target_val, pattern=best_pattern
    )
    r = backtest(strat, chunk, bankroll_cfg)

    mc_profits.append(r.total_profit)
    mc_max_dds.append(r.max_drawdown_pct)
    mc_final_bankrolls.append(r.final_bankroll)
    mc_win_rates.append(r.win_rate)
    mc_bets_count.append(r.total_bets)

    if r.final_bankroll <= 0:
        ruin_count += 1

    if (sim + 1) % 2000 == 0:
        print(f"  {sim+1:,}/{n_simulations:,}")

mc_profits = np.array(mc_profits)
mc_max_dds = np.array(mc_max_dds)
mc_final_bankrolls = np.array(mc_final_bankrolls)
mc_win_rates = np.array(mc_win_rates)

print(f"\nSimulação completa!")

In [ ]:
# Relatório Monte Carlo
print("=" * 60)
print("RELATÓRIO MONTE CARLO")
print("=" * 60)
print(f"")
print(f"Simulações:        {n_simulations:,}")
print(f"Rounds/sessão:     {sim_session_size:,}")
print(f"")
print(f"--- LUCRO ---")
print(f"  Média:           R${mc_profits.mean():+.2f}")
print(f"  Mediana:         R${np.median(mc_profits):+.2f}")
print(f"  Desvio Padrão:   R${mc_profits.std():.2f}")
print(f"  Percentil 5%:    R${np.percentile(mc_profits, 5):+.2f}")
print(f"  Percentil 25%:   R${np.percentile(mc_profits, 25):+.2f}")
print(f"  Percentil 75%:   R${np.percentile(mc_profits, 75):+.2f}")
print(f"  Percentil 95%:   R${np.percentile(mc_profits, 95):+.2f}")
print(f"")
print(f"--- RISCO ---")
print(f"  Risk of Ruin:    {ruin_count/n_simulations*100:.2f}% ({ruin_count}/{n_simulations})")
print(f"  % Sessões +:     {(mc_profits > 0).mean()*100:.1f}%")
print(f"  % Sessões -:     {(mc_profits <= 0).mean()*100:.1f}%")
print(f"  Max Drawdown Médio: {mc_max_dds.mean():.1f}%")
print(f"  Max Drawdown P95:   {np.percentile(mc_max_dds, 95):.1f}%")
print(f"")
print(f"--- ATIVIDADE ---")
print(f"  Win Rate Média:  {mc_win_rates[mc_win_rates > 0].mean():.1f}%")
print(f"  Apostas/sessão:  {np.mean(mc_bets_count):.0f}")
print(f"")
print(f"--- INTERVALO DE CONFIANÇA 95% ---")
ci_low = np.percentile(mc_profits, 2.5)
ci_high = np.percentile(mc_profits, 97.5)
print(f"  Lucro: [R${ci_low:+.2f}, R${ci_high:+.2f}]")

In [ ]:
# Visualização Monte Carlo
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Distribuição do lucro
axes[0, 0].hist(mc_profits, bins=80, color=COLORS['blue'], alpha=0.7, edgecolor='black', linewidth=0.3)
axes[0, 0].axvline(x=0, color=COLORS['red'], linestyle='--', linewidth=2, label='Break-even')
axes[0, 0].axvline(x=mc_profits.mean(), color=COLORS['green'], linestyle='-', linewidth=2, label=f'Média: R${mc_profits.mean():+.0f}')
axes[0, 0].axvline(x=np.median(mc_profits), color=COLORS['yellow'], linestyle='-.', linewidth=2, label=f'Mediana: R${np.median(mc_profits):+.0f}')
axes[0, 0].set_title(f'Distribuição do Lucro ({n_simulations:,} simulações)')
axes[0, 0].set_xlabel('Lucro (R$)')
axes[0, 0].set_ylabel('Frequência')
axes[0, 0].legend(fontsize=9)

# Distribuição do max drawdown
axes[0, 1].hist(mc_max_dds, bins=60, color=COLORS['red'], alpha=0.7, edgecolor='black', linewidth=0.3)
axes[0, 1].axvline(x=mc_max_dds.mean(), color=COLORS['yellow'], linestyle='-', linewidth=2, label=f'Média: {mc_max_dds.mean():.1f}%')
axes[0, 1].axvline(x=np.percentile(mc_max_dds, 95), color=COLORS['red'], linestyle='--', linewidth=2, label=f'P95: {np.percentile(mc_max_dds, 95):.1f}%')
axes[0, 1].set_title('Distribuição do Max Drawdown')
axes[0, 1].set_xlabel('Max Drawdown (%)')
axes[0, 1].set_ylabel('Frequência')
axes[0, 1].legend(fontsize=9)

# Win rate distribution
valid_wr = mc_win_rates[mc_win_rates > 0]
axes[1, 0].hist(valid_wr, bins=50, color=COLORS['green'], alpha=0.7, edgecolor='black', linewidth=0.3)
axes[1, 0].axvline(x=valid_wr.mean(), color=COLORS['yellow'], linestyle='-', linewidth=2, label=f'Média: {valid_wr.mean():.1f}%')
axes[1, 0].set_title('Distribuição da Win Rate')
axes[1, 0].set_xlabel('Win Rate (%)')
axes[1, 0].set_ylabel('Frequência')
axes[1, 0].legend(fontsize=9)

# Percentis cumulativos do lucro
sorted_profits = np.sort(mc_profits)
cumulative = np.arange(1, len(sorted_profits) + 1) / len(sorted_profits) * 100
axes[1, 1].plot(sorted_profits, cumulative, color=COLORS['cyan'], linewidth=1.5)
axes[1, 1].axvline(x=0, color=COLORS['red'], linestyle='--', alpha=0.5)
axes[1, 1].axhline(y=50, color=COLORS['dim'], linestyle=':', alpha=0.5)
axes[1, 1].fill_betweenx(cumulative, sorted_profits, 0,
                          where=sorted_profits < 0, color=COLORS['red'], alpha=0.1)
axes[1, 1].fill_betweenx(cumulative, 0, sorted_profits,
                          where=sorted_profits >= 0, color=COLORS['green'], alpha=0.1)
axes[1, 1].set_title('CDF do Lucro (Probabilidade Cumulativa)')
axes[1, 1].set_xlabel('Lucro (R$)')
axes[1, 1].set_ylabel('% das Simulações')
pct_positive = (mc_profits > 0).mean() * 100
axes[1, 1].text(0.02, 0.95, f'{pct_positive:.1f}% das sessões\nsão lucrativas',
                transform=axes[1, 1].transAxes, fontsize=10,
                color=COLORS['green'], verticalalignment='top')

plt.tight_layout()
plt.show()

## 6. Análise por Regime Temporal

Performance segmentada por mês para verificar estabilidade.

In [ ]:
# Performance mensal
df['month'] = df['date'].dt.strftime('%Y-%m')
months = df['month'].unique()

monthly_results = []

for month in months:
    month_data = df[df['month'] == month]['multiplicador'].values
    if len(month_data) < 500:  # Ignorar meses com poucos dados
        continue

    strat = MartingaleStrategy(
        trigger=best_trigger, target=best_target_val, pattern=best_pattern
    )
    r = backtest(strat, month_data, bankroll_cfg)

    monthly_results.append({
        'month': month,
        'rounds': len(month_data),
        'profit': r.total_profit,
        'profit_pct': r.total_profit_pct,
        'bets': r.total_bets,
        'win_rate': r.win_rate,
        'max_dd': r.max_drawdown_pct,
        'sharpe': r.sharpe_ratio,
        'pct_low': (month_data < LOW_THRESHOLD).mean() * 100,
    })

monthly_df = pd.DataFrame(monthly_results)

print(f"Meses analisados: {len(monthly_df)}")
print(f"Meses lucrativos: {(monthly_df['profit'] > 0).sum()} ({(monthly_df['profit'] > 0).mean()*100:.0f}%)")
print(f"Lucro médio/mês:  R${monthly_df['profit'].mean():+.2f}")
print(f"Desvio padrão:    R${monthly_df['profit'].std():.2f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

# Lucro mensal (barras)
colors = [COLORS['green'] if p > 0 else COLORS['red'] for p in monthly_df['profit']]
axes[0].bar(range(len(monthly_df)), monthly_df['profit'], color=colors, alpha=0.8)
axes[0].axhline(y=0, color=COLORS['dim'], linestyle='-', alpha=0.3)
axes[0].axhline(y=monthly_df['profit'].mean(), color=COLORS['yellow'],
                linestyle='--', alpha=0.7, label=f"Média: R${monthly_df['profit'].mean():+.0f}")
axes[0].set_title('Lucro Mensal')
axes[0].set_ylabel('R$')
axes[0].legend()
# Mostrar labels a cada 3 meses
tick_positions = range(0, len(monthly_df), 3)
tick_labels = [monthly_df.iloc[i]['month'] for i in tick_positions]
axes[0].set_xticks(list(tick_positions))
axes[0].set_xticklabels(tick_labels, rotation=45, fontsize=8, ha='right')

# % LOW por mês vs lucro
ax2 = axes[1].twinx()
axes[1].bar(range(len(monthly_df)), monthly_df['pct_low'],
            color=COLORS['blue'], alpha=0.3, label='% LOW')
ax2.plot(range(len(monthly_df)), monthly_df['profit'],
         color=COLORS['green'], linewidth=1.5, marker='.', markersize=4, label='Lucro')
axes[1].axhline(y=54.5, color=COLORS['dim'], linestyle='--', alpha=0.5, label='Média histórica (54.5%)')
axes[1].set_title('% LOW por Mês vs Lucro')
axes[1].set_ylabel('% LOW')
ax2.set_ylabel('Lucro R$')
axes[1].legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)
axes[1].set_xticks(list(tick_positions))
axes[1].set_xticklabels(tick_labels, rotation=45, fontsize=8, ha='right')

# Lucro acumulado
cumulative_profit = monthly_df['profit'].cumsum()
axes[2].plot(range(len(monthly_df)), cumulative_profit,
             color=COLORS['cyan'], linewidth=2)
axes[2].fill_between(range(len(monthly_df)), 0, cumulative_profit,
                     where=cumulative_profit >= 0, color=COLORS['green'], alpha=0.1)
axes[2].fill_between(range(len(monthly_df)), 0, cumulative_profit,
                     where=cumulative_profit < 0, color=COLORS['red'], alpha=0.1)
axes[2].axhline(y=0, color=COLORS['dim'], linestyle='-', alpha=0.3)
axes[2].set_title('Lucro Acumulado Mensal')
axes[2].set_ylabel('R$ Acumulado')
axes[2].set_xlabel('Mês')
axes[2].set_xticks(list(tick_positions))
axes[2].set_xticklabels(tick_labels, rotation=45, fontsize=8, ha='right')

plt.tight_layout()
plt.show()

## 7. Stress Test: Piores Cenários

Identificar os períodos mais hostis e verificar a sobrevivência.

In [ ]:
# Encontrar os 10 piores blocos contíguos de 2000 rounds
block_size = 2000
worst_blocks = []

for start in range(0, len(multipliers) - block_size, block_size // 4):
    chunk = multipliers[start:start + block_size]
    pct_low = (chunk < LOW_THRESHOLD).mean()
    worst_blocks.append((start, pct_low))

worst_blocks.sort(key=lambda x: x[1], reverse=True)

print("STRESS TEST: 10 Piores Blocos de 2000 Rounds")
print("=" * 80)
print(f"{'Bloco':>8} {'Início':>10} {'%LOW':>7} {'Lucro':>10} {'DD%':>7} {'Apostas':>8} {'WR%':>6} {'Sobrevive':>10}")
print("-" * 80)

stress_results = []
for i, (start, pct_low) in enumerate(worst_blocks[:10]):
    chunk = multipliers[start:start + block_size]
    strat = MartingaleStrategy(
        trigger=best_trigger, target=best_target_val, pattern=best_pattern
    )
    r = backtest(strat, chunk, bankroll_cfg)

    survived = "SIM" if r.final_bankroll > 0 else "NAO"
    stress_results.append(r)

    print(f"{i+1:>8} {start:>10,} {pct_low*100:>6.1f}% R${r.total_profit:>+8.0f} "
          f"{r.max_drawdown_pct:>6.1f}% {r.total_bets:>8} {r.win_rate:>5.0f}% {survived:>10}")

# Plot piores cenários
fig, ax = plt.subplots(figsize=(16, 6))
for i, r in enumerate(stress_results[:5]):
    start = worst_blocks[i][0]
    pct = worst_blocks[i][1]
    c = colors_list[i % len(colors_list)]
    label = f"Bloco {start:,} ({pct*100:.1f}% LOW) → R${r.total_profit:+.0f}"
    ax.plot(r.equity_curve, color=c, linewidth=1.2, alpha=0.8, label=label)
ax.axhline(y=1000, color=COLORS['dim'], linestyle='--', alpha=0.3)
ax.set_title('Equity Curve nos 5 Piores Cenários Históricos')
ax.set_ylabel('Banca (R$)')
ax.set_xlabel('Round')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 8. Compound Optimization com Config Final

Testar reinvestimento parcial sobre a melhor config.

In [ ]:
# Compound vs Flat para a config final
compound_configs = [
    ('Flat (0%)', 0.0, False),
    ('Compound 25%', 0.25, True),
    ('Compound 50%', 0.50, True),
    ('Compound 75%', 0.75, True),
    ('Compound 100%', 1.00, True),
]

compound_results = []
print("COMPOUND vs FLAT (Dados Completos)")
print("=" * 90)

for name, cpct, compound in compound_configs:
    cfg = BankrollConfig(
        initial_bankroll=1000,
        base_bet_pct=0.0167,
        compound=compound,
        compound_pct=cpct,
        stop_loss_pct=best_sl_val,
        stop_gain_pct=10.0,  # Sem stop gain para ver crescimento
    )
    strat = MartingaleStrategy(
        trigger=best_trigger, target=best_target_val, pattern=best_pattern
    )
    r = backtest(strat, multipliers, cfg)
    r.strategy_name = name
    compound_results.append(r)
    print(f"{name:20s} | Final: R${r.final_bankroll:>12,.2f} | "
          f"DD: {r.max_drawdown_pct:>5.1f}% | Apostas: {r.total_bets:>6}")

plot_comparison(compound_results, 'Compound Optimization - Config Final')

## 9. Configuração Final - Pronta para o Bot

Resumo de todos os achados em 3 perfis de risco.

In [ ]:
# Gerar configuração final baseada nos resultados
bwf_config = best_wf['config']

# Monte Carlo stats
mc_mean = mc_profits.mean()
mc_median = np.median(mc_profits)
mc_ci_low = np.percentile(mc_profits, 2.5)
mc_ci_high = np.percentile(mc_profits, 97.5)
mc_ruin_pct = ruin_count / n_simulations * 100
mc_positive_pct = (mc_profits > 0).mean() * 100

print("" + "=" * 70)
print("  ESTRATÉGIA FINAL OTIMIZADA - CRASH BOT")
print("=" * 70)
print()
print(f"  Base: {len(multipliers):,} rounds analisados (Dez/2022 - Fev/2026)")
print(f"  Validação: Walk-forward (treino 2022-2024, teste 2025-2026)")
print(f"  Robustez: {n_simulations:,} simulações Monte Carlo")
print()
print(f"  --- PARÂMETROS ÓTIMOS ---")
print(f"  Trigger:          {best_trigger} LOWs consecutivos")
print(f"  Pattern:          {bwf_config['pattern']}")
print(f"  Target:           {best_target_val}x")
print(f"  Stop Loss:        {best_sl_val*100:.0f}% da banca")
print(f"  Stop Gain:        {best_sg_val*100:.0f}% da banca")
print(f"  Base Bet:         1.67% da banca (banca/6)")
print()
print(f"  --- MÉTRICAS VALIDADAS (out-of-sample) ---")
print(f"  Sharpe (treino):  {best_wf['train_sharpe']:+.4f}")
print(f"  Sharpe (teste):   {best_wf['test_sharpe']:+.4f}")
print(f"  Win Rate:         {best_wf['test_wr']:.0f}%")
print(f"  Max Drawdown:     {best_wf['test_dd']:.1f}%")
print()
print(f"  --- MONTE CARLO ({n_simulations:,} simulações) ---")
print(f"  Lucro médio/sessão: R${mc_mean:+.2f}")
print(f"  IC 95%:           [R${mc_ci_low:+.2f}, R${mc_ci_high:+.2f}]")
print(f"  % Sessões +:      {mc_positive_pct:.1f}%")
print(f"  Risk of Ruin:     {mc_ruin_pct:.2f}%")
print()

# Perfis de risco
print("  --- PERFIS DE RISCO RECOMENDADOS ---")
print()

profiles = {
    'CONSERVADOR': {
        'trigger': max(best_trigger, 7),
        'pattern': '1/2',
        'target': best_target_val,
        'base_bet_pct': 0.01,
        'stop_gain_pct': 0.10,
        'stop_loss_pct': 0.20,
        'compound': False,
        'compound_pct': 0.0,
        'saque_pct': 1.0,
        'desc': 'Baixo risco. Ganhos pequenos e consistentes.',
    },
    'MODERADO': {
        'trigger': best_trigger,
        'pattern': bwf_config['pattern'],
        'target': best_target_val,
        'base_bet_pct': 0.0167,
        'stop_gain_pct': best_sg_val,
        'stop_loss_pct': best_sl_val,
        'compound': True,
        'compound_pct': 0.50,
        'saque_pct': 0.50,
        'desc': 'Balanço risco/retorno. Config otimizada pelo grid search.',
    },
    'AGRESSIVO': {
        'trigger': max(best_trigger - 1, 5),
        'pattern': '1/2/4/8',
        'target': best_target_val,
        'base_bet_pct': 0.0167,
        'stop_gain_pct': 0.30,
        'stop_loss_pct': 0.80,
        'compound': True,
        'compound_pct': 0.75,
        'saque_pct': 0.25,
        'desc': 'Alto risco. Potencial de crescimento rápido, mas maior drawdown.',
    },
}

for pname, p in profiles.items():
    print(f"  {pname}:")
    print(f"    Trigger: T{p['trigger']} | Pattern: {p['pattern']} | Target: {p['target']}x")
    print(f"    Bet: {p['base_bet_pct']*100:.2f}% | SG: {p['stop_gain_pct']*100:.0f}% | SL: {p['stop_loss_pct']*100:.0f}%")
    cmp = f"{p['compound_pct']*100:.0f}%" if p['compound'] else 'Não'
    print(f"    Compound: {cmp} | Saque: {p['saque_pct']*100:.0f}%")
    print(f"    → {p['desc']}")
    print()

## 10. Exportar Configuração para JSON

In [ ]:
# Exportar config final como JSON para uso no bot
output = {
    'metadata': {
        'generated_by': 'NB-05 Final Strategy',
        'data_range': 'Dec/2022 - Feb/2026',
        'total_rounds': int(len(multipliers)),
        'validation': 'walk-forward (train 2022-2024, test 2025-2026)',
        'monte_carlo_sims': n_simulations,
    },
    'optimal_params': {
        'trigger': best_trigger,
        'pattern': bwf_config['pattern'],
        'target': best_target_val,
        'base_bet_pct': 0.0167,
        'stop_gain_pct': best_sg_val,
        'stop_loss_pct': best_sl_val,
    },
    'monte_carlo': {
        'mean_profit': round(float(mc_mean), 2),
        'median_profit': round(float(mc_median), 2),
        'ci_95_low': round(float(mc_ci_low), 2),
        'ci_95_high': round(float(mc_ci_high), 2),
        'pct_positive_sessions': round(float(mc_positive_pct), 1),
        'risk_of_ruin_pct': round(float(mc_ruin_pct), 2),
        'max_dd_mean': round(float(mc_max_dds.mean()), 1),
        'max_dd_p95': round(float(np.percentile(mc_max_dds, 95)), 1),
    },
    'walk_forward': {
        'train_sharpe': round(float(best_wf['train_sharpe']), 4),
        'test_sharpe': round(float(best_wf['test_sharpe']), 4),
        'train_profit': round(float(best_wf['train_profit']), 2),
        'test_profit': round(float(best_wf['test_profit']), 2),
    },
    'profiles': profiles,
}

output_path = Path('..') / 'data' / 'processed' / 'final_strategy.json'
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print(f"Config exportada para: {output_path.resolve()}")
print()
print(json.dumps(output, indent=2, ensure_ascii=False))

In [ ]:
print("=" * 60)
print("PIPELINE COMPLETO")
print("=" * 60)
print()
print("NB-01: Perfil estatístico do jogo")
print("NB-02: Framework de backtesting")
print("NB-03: Torneio de 9 estratégias")
print("NB-04: Gestão de banca e compound")
print("NB-05: Estratégia final otimizada (este notebook)")
print()
print(f"Resultado: config exportada para data/processed/final_strategy.json")
print(f"Próximo passo: carregar config no bot e validar em tempo real")